In [1]:
import warnings
from IPython.core.interactiveshell import InteractiveShell
from IPython.display import display, Markdown, HTML, Javascript
from tqdm import tqdm


def alert(message='Loop completed!'):
    # Check if the system supports 'say' and notifications
    os.system(f'osascript -e \'display notification "{message}" with title "Notification"\'')
    # Speak the alert
    os.system(f'say "{message}"')        
        
InteractiveShell.ast_node_interactivity = "all"
warnings.filterwarnings("ignore")
tqdm.pandas()

In [2]:
import geopandas as gpd
import pandas as pd

In [29]:
area_df = gpd.read_file('data/raw/areas.shp')
area_df['latlng'] = area_df.lookup_lat.fillna(area_df.center_lat).apply(eval)
area_df = area_df.rename(columns={'intersecti': 'intersection_id'})
# we first chopped it to test area_df = area_df.head()

In [49]:
print(area_df.tail(10))

       intersection_id    zip county district  population          state  \
60158            65430  25674  Wayne       01      1771.0  West Virginia   
60159            65431  25524  Wayne       01      2786.0  West Virginia   
60160            65432  25530  Wayne       05      6665.0  West Virginia   
60161            65434  25530  Wayne       01      6665.0  West Virginia   
60162            65435  25535  Wayne       01      2991.0  West Virginia   
60163            65436  25555  Wayne       05      2518.0  West Virginia   
60164            65437  25555  Wayne       01      2518.0  West Virginia   
60165            65438  25570  Wayne       01      4887.0  West Virginia   
60166            65439  25699  Wayne       01       606.0  West Virginia   
60167            65440  25701  Wayne       01     21375.0  West Virginia   

      state_id                                center_lat  \
60158       WV   (37.89816159256202, -82.38922280324576)   
60159       WV   (38.03135364993199, -82.22

In [31]:
area_df['district'] = area_df.district.str.replace('00', '01')

In [15]:
#we imported results results_2024-11-03.csv to see what was inside 
old_result_df = pd.read_csv('data/raw/results_2024-11-03.csv')
old_result_df.head()

,Unnamed: 0,zip,county,district,state,lat,lng,response
0,0,68791,Cuming,NE-CD01,Nebraska,41.991030,-96.932450,"{""success"": true, ""data"": {""districts"": [{""id""..."
1,21,68047,Cuming,NE-CD01,Nebraska,42.062633,-96.755579,"{""success"": true, ""data"": {""districts"": [{""id""..."
2,40,68057,Cuming,NE-CD01,Nebraska,41.748409,-96.593060,"{""success"": true, ""data"": {""districts"": [{""id""..."
3,61,68038,Cuming,NE-CD01,Nebraska,41.913534,-96.571270,"{""success"": true, ""data"": {""districts"": [{""id""..."
4,82,68641,Cuming,NE-CD01,Nebraska,41.802225,-96.993344,"{""success"": true, ""data"": {""districts"": [{""id""..."


# Define Ballotpedia API lookup

Caches and retrieves Ballotpedia geographic data for a given latitude/longitude. Uses polite rate limiting and request headers. Function has a 100,000 item cache to avoid duplicate API calls.

In [9]:
import requests
from functools import lru_cache

@lru_cache(maxsize=100_000)
def get_ballotpedia_data_rigorous(lat, lng, rate_limit=2):
    url = "https://api4.ballotpedia.org/myvote_redistricting_with_historical"
    params = {
        'long': str(lng),
        'lat': str(lat),
        'include_volunteer': 'true'
    }
    headers = {
        'Accept': 'application/json',
        'Content-Type': 'application/json',
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36',
        'Referer': 'https://sblv3.ballotpedia.org/',
        'Origin': 'https://sblv3.ballotpedia.org'
    }
    
    response = requests.get(url, params=params, headers=headers)

    # Rate limit
    time.sleep(rate_limit)
    
    if response.json().get('message') == 'Forbidden':
        raise PermissionError


    print(response.json())
    return response.json()
    
  
    

# Define processing functions

Functions to transform raw API election data into a structured format:
- Process ballot measures and candidate information
- Calculate decision metrics (number of races and options)
- Format final output as a pandas DataFrame with location data

In [10]:
import pandas as pd
from typing import Dict, List, Optional
from collections import defaultdict

def extract_election_data(api_response: Dict) -> List[Dict]:
    return api_response.get('data', {}).get('elections', [])

def process_ballot_measure(measure: Dict, common_data: Dict) -> Dict:
    return {
        **common_data,
        'race_type': 'Ballot Measure',
        'office.name': measure['name'],
        'office.type': 'Ballot Measure',
        'office.level': common_data['district_type'],
        'office.branch': 'N/A',
        'number_of_seats': 1,
        'person.name': 'Yes/No Question',
        'person.url': None,
        'party_affiliation': None,
        'status': 'On the Ballot',
        'is_incumbent': False,
        'running_mate.name': None,
        'measure_id': measure['id'],
        'measure_district_type': measure['district_type']
    }

def process_candidate(candidate: Dict, race: Dict, common_data: Dict) -> Dict:
    office = race['office']
    return {
        **common_data,
        'race_type': 'Candidate',
        'office.name': office['name'],
        'office.type': office['type'],
        'office.level': office['level'],
        'office.branch': office['branch'],
        'number_of_seats': race['number_of_seats'],
        'person.name': candidate['person']['name'],
        'person.url': candidate['person']['url'],
        'party_affiliation': candidate['party_affiliation'],
        'status': candidate['status'],
        'is_incumbent': candidate['is_incumbent'],
        'running_mate.name': candidate['running_mate']['name'] if candidate.get('running_mate') else None,
        'measure_id': None,
        'measure_district_type': None
    }

def process_district(district: Dict, election_date: str) -> List[Dict]:
    common_data = {
        'election_date': election_date,
        'district_name': district['name'],
        'district_type': district['type']
    }
    
    ballot_measures = [process_ballot_measure(measure, common_data) for measure in district.get('ballot_measures') or []]
    candidates = [process_candidate(candidate, race, common_data) 
                  for race in district.get('races')  or []
                  for candidate in race['candidates']]
    
    return ballot_measures + candidates

def calculate_decision_metrics(df: pd.DataFrame) -> pd.DataFrame:
    def count_decisions_and_options(group):
        decisions = defaultdict(int)
        for _, row in group.iterrows():
            if row['race_type'] == 'Ballot Measure':
                decisions[row['office.name']] = 2  # Yes/No options
            else:
                decisions[row['office.name']] += 1
        
        unique_decisions = len(decisions)
        total_options = sum(decisions.values())
        
        group['unique_decisions'] = unique_decisions
        group['total_options'] = total_options
        return group

    return df.groupby(['district_name', 'district_type']).apply(count_decisions_and_options).reset_index(drop=True)

def process_api_response(api_response: Dict, lat: float, lng: float) -> Optional[pd.DataFrame]:
    if not api_response:
        return None
        
    elections = extract_election_data(api_response)
    if not elections:
        return None
    
    processed_data = [
        item for election in elections
        for district in election['districts']
        for item in process_district(district, election['date'])
    ]
    
    df = pd.DataFrame(processed_data)
    df['group_id'] = df.groupby(['district_name', 'district_type']).ngroup()
    df['lat'], df['lng'] = lat, lng
    
    df = calculate_decision_metrics(df)
    
    return df

# Define main processing loop

Processes and accumulates data for each geographic point through the Ballotpedia API

In [17]:
#running this cell to CREATE an empty result_df 
import json
#result_df = pd.read_csv('data/raw/results_2024-11-03.csv')
result_df = pd.DataFrame()

In [ ]:
#strategy: Eric is re-using previous data (collected during the elections) instead of creating a new db for the full ballot version; we must try from scratch data-wise, but using some of the methods already created

In [ ]:
# # Define the desired column names
# column_names = ['Name', 'Age', 'City']

# # Create an empty DataFrame with the specified columns
# empty_df = pd.DataFrame(columns=column_names)

# # Print the empty DataFrame
# print(empty_df)

In [32]:
# Define the desired column names
column_names = ['zip', 'state', 'district', 'county', 'lat', 'lng', 'response']

# Create an empty DataFrame with the specified columns
empty_df = pd.DataFrame(columns=column_names)

result_df = empty_df
# Print the empty DataFrame
print(result_df)

Empty DataFrame
Columns: [zip, state, district, county, lat, lng, response]
Index: []


In [26]:
# result_df['zip'] = result_df.zip.astype(str)
# result_df['district'] = result_df.district.astype(str).str.slice(-2)
# result_df['lat'] = result_df.lat.astype(str).str.slice(0, 12)
# result_df['lng'] = result_df.lng.astype(str).str.slice(0, 12)
response_lookup = dict(result_df
                       .drop_duplicates(['zip', 'lat', 'lng'])
                       .set_index(['zip', 'lat', 'lng'])
                       .response.apply(json.loads))
response_lookup.update(dict(result_df
                            .drop_duplicates(['zip', 'county', 'district'])
                            .set_index(['zip', 'county', 'district'])
                            .response.apply(json.loads)))

In [44]:
import os
import time

def process_dataframe(df):
    all_results = []
    
    for row in tqdm(df.itertuples(), total=len(df), desc="Processing rows"):

        lat, lng = row.latlng
        key1 = (str(int(row.zip)), str(lat)[:12], str(lng)[:12])
        key2 = (f'{int(row.zip):05}', str(lat)[:12], str(lng)[:12])
        key3 = (str(int(row.zip)), row.county, row.district)
        key4 = (str(int(row.zip)), row.county, None)
        key5 = (str(int(row.zip)), row.county, 'ne')
        key6 = (str(int(row.zip)), row.county, '00')
        key7 = (str(int(row.zip)), row.county, '01')
        for key_ in [key1, key2, key3, key4, key5, key6, key7]:
            if key_ in response_lookup:
                break
        key = key_
        if key in response_lookup:
            response = response_lookup[key]
        else:
            response = None
            for i in range(60):
                try:
                    response = get_ballotpedia_data_rigorous(lat, lng, rate_limit=0.1)
                    if response['data']['districts'] is not None:
                        response['data']['districts'][0]
                    break
                except PermissionError:
                    if i == 0:
                        print('PermissionError:', lat, lng, row.state, row.county)
                    os.system('say "beep"')  # Uses system text-to-speech to make a sound
                    time.sleep(3**(i))
                    continue
                except Exception as e:
                    print('Exception:', e, lat, lng, row.state, row.county)
                    print(response)
                    print(e)
                    time.sleep(2**(i))
                    continue
                
        response_df = process_api_response(response, lat, lng)
        if response_df is None:
            display(row.zip)
            display(row.county)
            display(row.district)
            continue
        else:
            response_lookup[row.zip, row.county, row.district] = response
            
        response_df['response'] = json.dumps(response)
        response_df['intersection_id'] = row.intersection_id
        response_df['district'] = row.district
        response_df['county'] = row.county
        response_df['state'] = row.state
        response_df['zip'] = row.zip
        response_df['geometry'] = row.geometry
        results = response_df.to_dict(orient='records')
        
        all_results.extend(results)
    # return all_results (failed attempt 9/22)
        
    return pd.DataFrame(all_results)

In [50]:
sample_area_df = area_df.iloc[120:170]

In [51]:
sample_area_df

,intersection_id,zip,county,district,population,state,state_id,center_lat,lookup_lat,best_width,link,geometry,latlng
120,140,95936,Sierra,03,146.0,California,CA,"(39.56334083842082, -120.7928168166203)","(39.56336, -120.79279)",6243.286785,https://edbltn.github.io/show-me-the-ballot/da...,"POLYGON ((-2095264.521 2111470.261, -2095257.5...","(39.56336, -120.79279)"
121,141,95960,Sierra,03,647.0,California,CA,"(39.43397267406749, -120.9724979006489)","(39.43397267406749, -120.9724979006489)",3287.766034,https://edbltn.github.io/show-me-the-ballot/da...,"POLYGON ((-2103559.932 2099602.534, -2103574.1...","(39.43397267406749, -120.9724979006489)"
122,142,95944,Sierra,03,647.0,California,CA,"(39.53484448258896, -120.86151275525232)","(39.53482, -120.86159)",2835.936694,https://edbltn.github.io/show-me-the-ballot/da...,"POLYGON ((-2096173.573 2113442.189, -2096151.3...","(39.53482, -120.86159)"
123,145,38549,Clinton,01,4115.0,Tennessee,TN,"(36.62904160864791, -85.15907395015846)","(36.62904160864791, -85.15907395015846)",926.549184,https://edbltn.github.io/show-me-the-ballot/da...,"MULTIPOLYGON (((957477.504 1563339.03, 957349....","(36.62904160864791, -85.15907395015846)"
124,146,42717,Clinton,01,5953.0,Kentucky,KY,"(36.67011781873429, -85.26428368249475)","(36.67011781873429, -85.26428368249475)",2164.947426,https://edbltn.github.io/show-me-the-ballot/da...,"MULTIPOLYGON (((948198.685 1562503.111, 948112...","(36.67011781873429, -85.26428368249475)"
125,147,42717,Clinton,06,5953.0,Kentucky,KY,"(36.62603647734905, -85.29535459873286)","(36.62603647734905, -85.29535459873286)",34.905863,https://edbltn.github.io/show-me-the-ballot/da...,"MULTIPOLYGON (((946836.971 1562308.294, 946853...","(36.62603647734905, -85.29535459873286)"
126,148,42602,Clinton,05,9255.0,Kentucky,KY,"(36.7163442528393, -85.00994673876218)","(36.7163442528393, -85.00994673876218)",515.330287,https://edbltn.github.io/show-me-the-ballot/da...,"MULTIPOLYGON (((973730.479 1564660.914, 973744...","(36.7163442528393, -85.00994673876218)"
127,149,42602,Clinton,01,9255.0,Kentucky,KY,"(36.72938186345639, -85.13375009209312)","(36.73201, -85.13324)",17831.086030,https://edbltn.github.io/show-me-the-ballot/da...,"POLYGON ((969991.06 1577959.949, 969986.14 157...","(36.73201, -85.13324)"
128,150,42602,Clinton,06,9255.0,Kentucky,KY,"(36.62238971615073, -85.09630313505714)","(36.62238971615073, -85.09630313505714)",16.112004,https://edbltn.github.io/show-me-the-ballot/da...,"POLYGON ((964046.553 1563809.657, 963984.565 1...","(36.62238971615073, -85.09630313505714)"
129,151,42603,Clinton,05,521.0,Kentucky,KY,"(36.78850355924633, -85.02057487135353)","(36.78850355924633, -85.02057487135353)",455.433255,https://edbltn.github.io/show-me-the-ballot/da...,"POLYGON ((967814.078 1585001.896, 967815.594 1...","(36.78850355924633, -85.02057487135353)"


In [52]:
result_df = process_dataframe(sample_area_df)

Processing rows:   2%|█                                                  | 1/50 [00:00<00:32,  1.52it/s]

{'success': True, 'data': {'districts': [{'id': 516, 'name': 'California', 'type': 'State', 'state': 'CA'}], 'polling_place_locator': {'url': 'https://www.sos.ca.gov/elections/polling-place/'}, 'elections': [{'date': '2025-11-04', 'candidate_lists_complete': False, 'districts': [{'id': 516, 'name': 'California', 'type': 'State', 'precise_boundary': True, 'disclaimers': None, 'permanent_disclaimers': None, 'ballot_measures': [{'id': 28073, 'name': 'California Proposition 50, Use of Legislative Congressional Redistricting Map Amendment (2025)', 'district_type': 'State'}], 'races': None}]}]}, 'message': None}


Processing rows:   4%|██                                                 | 2/50 [00:01<00:25,  1.86it/s]

{'success': True, 'data': {'districts': [{'id': 516, 'name': 'California', 'type': 'State', 'state': 'CA'}], 'polling_place_locator': {'url': 'https://www.sos.ca.gov/elections/polling-place/'}, 'elections': [{'date': '2025-11-04', 'candidate_lists_complete': False, 'districts': [{'id': 516, 'name': 'California', 'type': 'State', 'precise_boundary': True, 'disclaimers': None, 'permanent_disclaimers': None, 'ballot_measures': [{'id': 28073, 'name': 'California Proposition 50, Use of Legislative Congressional Redistricting Map Amendment (2025)', 'district_type': 'State'}], 'races': None}]}]}, 'message': None}


Processing rows:   6%|███                                                | 3/50 [00:01<00:23,  2.01it/s]

...

{'success': True, 'data': {'districts': [{'id': 512, 'name': 'Alabama', 'type': 'State', 'state': 'AL'}], 'polling_place_locator': {'url': 'https://myinfo.alabamavotes.gov/VoterView/PollingPlaceSearch.do'}, 'elections': [{'date': '2026-05-19', 'candidate_lists_complete': False, 'districts': [{'id': 512, 'name': 'Alabama', 'type': 'State', 'precise_boundary': True, 'disclaimers': None, 'permanent_disclaimers': None, 'ballot_measures': [{'id': 27933, 'name': 'Alabama Allow Judges to Deny Bail for Certain Weapon Discharges and Solicitation, Attempt, or Conspiracy to Commit Murder Amendment (May 2026)', 'district_type': 'State'}, {'id': 27947, 'name': 'Alabama Prohibit Diminishing District Attorney Compensation During Term of Office Amendment (May 2026)', 'district_type': 'State'}], 'races': None}, {'id': 3, 'name': 'Alabama District 3', 'type': 'Congress', 'precise_boundary': True, 'disclaimers': None, 'permanent_disclaimers': [{'id': 1093, 'sort': 1, 'text': 'A three-judge panel of the U

Processing rows:  98%|█████████████████████████████████████████████████ | 49/50 [00:22<00:00,  2.27it/s]

{'success': True, 'data': {'districts': [{'id': 512, 'name': 'Alabama', 'type': 'State', 'state': 'AL'}], 'polling_place_locator': {'url': 'https://myinfo.alabamavotes.gov/VoterView/PollingPlaceSearch.do'}, 'elections': [{'date': '2026-05-19', 'candidate_lists_complete': False, 'districts': [{'id': 512, 'name': 'Alabama', 'type': 'State', 'precise_boundary': True, 'disclaimers': None, 'permanent_disclaimers': None, 'ballot_measures': [{'id': 27933, 'name': 'Alabama Allow Judges to Deny Bail for Certain Weapon Discharges and Solicitation, Attempt, or Conspiracy to Commit Murder Amendment (May 2026)', 'district_type': 'State'}, {'id': 27947, 'name': 'Alabama Prohibit Diminishing District Attorney Compensation During Term of Office Amendment (May 2026)', 'district_type': 'State'}], 'races': None}, {'id': 3, 'name': 'Alabama District 3', 'type': 'Congress', 'precise_boundary': True, 'disclaimers': None, 'permanent_disclaimers': [{'id': 1093, 'sort': 1, 'text': 'A three-judge panel of the U

Processing rows: 100%|██████████████████████████████████████████████████| 50/50 [00:23<00:00,  2.16it/s]

{'success': True, 'data': {'districts': [{'id': 512, 'name': 'Alabama', 'type': 'State', 'state': 'AL'}], 'polling_place_locator': {'url': 'https://myinfo.alabamavotes.gov/VoterView/PollingPlaceSearch.do'}, 'elections': [{'date': '2026-05-19', 'candidate_lists_complete': False, 'districts': [{'id': 512, 'name': 'Alabama', 'type': 'State', 'precise_boundary': True, 'disclaimers': None, 'permanent_disclaimers': None, 'ballot_measures': [{'id': 27933, 'name': 'Alabama Allow Judges to Deny Bail for Certain Weapon Discharges and Solicitation, Attempt, or Conspiracy to Commit Murder Amendment (May 2026)', 'district_type': 'State'}, {'id': 27947, 'name': 'Alabama Prohibit Diminishing District Attorney Compensation During Term of Office Amendment (May 2026)', 'district_type': 'State'}], 'races': None}, {'id': 3, 'name': 'Alabama District 3', 'type': 'Congress', 'precise_boundary': True, 'disclaimers': None, 'permanent_disclaimers': [{'id': 1093, 'sort': 1, 'text': 'A three-judge panel of the U

In [53]:
print(result_df)

     election_date district_name district_type       race_type  \
0       2025-11-04    California         State  Ballot Measure   
1       2025-11-04    California         State  Ballot Measure   
2       2025-11-04    California         State  Ballot Measure   
3       2026-05-19      Kentucky         State       Candidate   
4       2026-05-19      Kentucky         State       Candidate   
...            ...           ...           ...             ...   
1244    2026-05-19       Alabama         State  Ballot Measure   
1245    2026-05-19       Alabama         State  Ballot Measure   
1246    2026-05-19       Alabama         State  Ballot Measure   
1247    2026-05-19       Alabama         State  Ballot Measure   
1248    2026-05-19       Alabama         State  Ballot Measure   

                                            office.name     office.type  \
0     California Proposition 50, Use of Legislative ...  Ballot Measure   
1     California Proposition 50, Use of Legislative ...  

In [54]:
result_df['district'] = result_df.district.str.replace('00', '01')
import re

def extract_type_district_numbers(text, state):
    # Regular expression pattern to match <TYPE> District <NUMBER>
    pattern = f'U\\.S\\. House \\b{state}\\b (?:D|d)istrict (\\d+)'
    matches = re.findall(pattern, text)
    
    # Return matches as a list of tuples with <TYPE> and <NUMBER>
    return (matches or [None])[0]

In [56]:
state_id_lookup = dict(zip(area_df.state, area_df.state_id))

In [58]:
print(state_id_lookup)

{'Nebraska': 'NE', 'Washington': 'WA', 'New Mexico': 'NM', 'South Dakota': 'SD', 'Texas': 'TX', 'Nevada': 'NV', 'California': 'CA', 'Tennessee': 'TN', 'Kentucky': 'KY', 'Ohio': 'OH', 'Alabama': 'AL', 'Georgia': 'GA', 'Wisconsin': 'WI', 'Arkansas': 'AR', 'Oregon': 'OR', 'Pennsylvania': 'PA', 'Mississippi': 'MS', 'Missouri': 'MO', 'Colorado': 'CO', 'North Carolina': 'NC', 'Utah': 'UT', 'Oklahoma': 'OK', 'Virginia': 'VA', 'Wyoming': 'WY', 'West Virginia': 'WV', 'Louisiana': 'LA', 'New York': 'NY', 'Michigan': 'MI', 'Indiana': 'IN', 'Massachusetts': 'MA', 'Kansas': 'KS', 'Idaho': 'ID', 'Florida': 'FL', 'Alaska': 'AK', 'Illinois': 'IL', 'Vermont': 'VT', 'Montana': 'MT', 'Minnesota': 'MN', 'New Jersey': 'NJ', 'North Dakota': 'ND', 'Maryland': 'MD', 'Iowa': 'IA', 'South Carolina': 'SC', 'Maine': 'ME', 'Hawaii': 'HI', 'New Hampshire': 'NH', 'Arizona': 'AZ', 'Delaware': 'DE', 'District of Columbia': 'DC', 'Rhode Island': 'RI', 'Connecticut': 'CT'}


In [59]:
result_df['state_id'] = result_df.state.map(state_id_lookup)
result_df['district'] = result_df.progress_apply(
    lambda x: f'{x.state_id}-CD{int(str(x.district)[-2:]):02}'
    if x.district is not None and str(x.district)[-3:] != 'nan' else None, axis=1
)
result_df['extracted_district'] = result_df.progress_apply(
    lambda x: extract_type_district_numbers(x.response, x.state) or 0, axis=1
)
result_df['extracted_district'] = result_df.progress_apply(
    lambda x: f'{x.state_id}-CD{int(x.extracted_district):02}', axis=1
)

100%|████████████████████████████████████████████████████████████| 1249/1249 [00:00<00:00, 47015.35it/s]


In [60]:
result_df['district'] = result_df.district.fillna(result_df.extracted_district)

In [61]:
result_df['district'] 

0       CA-CD03
1       CA-CD03
2       CA-CD03
3       TN-CD01
4       TN-CD01
         ...   
1244    AL-CD03
1245    AL-CD03
1246    AL-CD03
1247    AL-CD03
1248    AL-CD03
Name: district, Length: 1249, dtype: object

In [62]:
result_df['extracted_district'] 

0       CA-CD00
1       CA-CD00
2       CA-CD00
3       TN-CD00
4       TN-CD00
         ...   
1244    AL-CD00
1245    AL-CD00
1246    AL-CD00
1247    AL-CD00
1248    AL-CD00
Name: extracted_district, Length: 1249, dtype: object

In [63]:
index = (result_df.district != result_df.extracted_district) & (~result_df.extracted_district.str.endswith('00'))
result_df.loc[index, 'extracted_district'].value_counts()
result_df.loc[index, 'district'] = result_df.loc[index, 'extracted_district']

extracted_district
KY-CD01    42
Name: count, dtype: int64

In [66]:
from datetime import date

today = date.today().strftime("%Y-%m-%d")

result_df[[
    'zip', 'county', 'district', 'state', 'lat', 'lng', 'response'
]].drop_duplicates(['zip', 'county', 'district']).to_csv(f'data/2025/results_{today}.csv')

In [18]:
alert('Complete!')

# Define ballot complexity analyzer

Function to analyze ballot complexity using GPT-4:
- Identifies non-partisan contests

Uses structured Pydantic models to validate GPT-4 outputs.

In [67]:
import json
import os
import re

from typing import List, Optional, Union, Literal
from pydantic import BaseModel, Field
from openai import OpenAI
from functools import lru_cache
from typing import List
from collections import Counter

# Initialize OpenAI client
# client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

class NonPartisanContest(BaseModel):
    is_non_partisan: bool

In [68]:
# Updated prompt for remaining indicators and analysis
SYSTEM_PROMPT = """
Determine if this is a non-partisan election contest.

Non-partisan contests typically:
- Don't list party affiliations
- Include local offices like school boards, city councils
- Include most judicial positions
- Are specifically designated as non-partisan

Partisan contests typically:
- List party affiliations
- Include races for Congress, President, Governor
- Are primary elections for political parties
- Include party committee positions

Provide analysis in JSON format:
{
    "is_non_partisan": boolean
}
"""
# Function to analyze a single ballot
@lru_cache(maxsize=100_000)
def analyze_race(race):
    
    # Create prompt for remaining analysis
    prompt = f"""
    Analyze the following race:
    {race}
    
    Provide the analysis in the specified structured JSON format.
    """
    
    try:
        completion = client.beta.chat.completions.parse(
            model="gpt-4o-mini",
            temperature=0.0,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt}
            ],
            response_format=NonPartisanContest
        )
        
        # Parse the JSON response
        analysis = json.loads(completion.choices[0].message.content)
        return analysis['is_non_partisan']
    except Exception as e:
        print(f"Error analyzing ballot: {e}")
        return None

# Define ballot formatting functions

Functions to create a markdown-formatted ballot display:
- Sorts races by importance (President → Federal → State → Local → Measures)
- Formats candidate details including party, incumbent status, and links
- Creates checkboxes for voting options

In [69]:
import pandas as pd

def format_and_display_ballot_info(group):
    # Sort the races
    sorted_group = sort_races(group)
    
    # Start building the markdown output
    markdown_output = f"# Ballot for {sorted_group['county'].iloc[0]} County, {sorted_group['state'].iloc[0]}\n\n"
    markdown_output += f"Election Date: {sorted_group['election_date'].iloc[0]}\n\n"

    # Process races by office
    outputs = []
    for office, office_group in sorted_group.groupby('office.name'):
        outputs.append((format_office(office, office_group), office_group.key.iloc[0]))

    outputs.sort(key=lambda x: x[1])
    for output, key in outputs:
        markdown_output += output

    return markdown_output

def sort_races(group):
    def race_priority(row):
        office = row['office.name'].lower()
        if row['race_type'] == 'Ballot Measure':
            return 5
        elif 'president' in office:
            return 0
        elif office.startswith('u.s.'):
            return 1
        elif 'governor' in office:
            return 2
        elif row['office.level'] == 'State':
            return 3
        else:
            return 4

    group['key'] = group.apply(race_priority, axis=1)
    group = group.sort_values(by='key')

    return group

def format_office(office, office_group):
    first_row = office_group.iloc[0]
    output = f"## {office}\n\n"
    
    if first_row['race_type'] == 'Ballot Measure':
        output += format_ballot_measure(first_row)
    else:
        output += format_candidate_race(office_group)
    
    output += "---\n\n"
    return output

def format_ballot_measure(measure):
    output = f"**Type:** Ballot Measure\n"
    output += f"**Level:** {measure['measure_district_type']}\n\n"
    return output

def format_candidate_race(race_group):
    first_row = race_group.iloc[0]
    output = f"**Level:** {first_row['office.level']}\n"
    output += f"**Branch:** {first_row['office.branch']}\n"
    output += f"**Number of Seats:** {first_row['number_of_seats']}\n\n"
    output += "### Candidates:\n"
    
    for _, candidate in race_group.iterrows():
        output += format_candidate(candidate)
    
    return output

def format_candidate(candidate):
    party = candidate['party_affiliation']
    party_name = party[0]['name'] if isinstance(party, list) and party else 'Unknown'
    
    output = f"- **{candidate['person.name']}** ({party_name})\n"
    bullets = []
    if pd.notna(candidate['running_mate.name']):
        bullets.append(f"Running Mate: {candidate['running_mate.name']}\n")
    if pd.notna(candidate['person.url']):
        bullets.append(f"[More Info]({candidate['person.url']})\n")
    if len(bullets) > 1:
        output += '    - ' + '    - '.join(bullets)
    else:
        output += ''.join(bullets)
    output += "\n"
    return output

In [70]:
for markdown_output in result_df.iloc[:100].groupby(['intersection_id', 'state']).apply(format_and_display_ballot_info):
    print(markdown_output)
    display(Markdown(markdown_output))
    break

# Ballot for Sierra County, California

Election Date: 2025-11-04

## California Proposition 50, Use of Legislative Congressional Redistricting Map Amendment (2025)

**Type:** Ballot Measure
**Level:** State

---




# Ballot for Sierra County, California

Election Date: 2025-11-04

## California Proposition 50, Use of Legislative Congressional Redistricting Map Amendment (2025)

**Type:** Ballot Measure
**Level:** State

---



In [71]:
import markdown
from bs4 import BeautifulSoup

def count_markdown_words(md_text):
    # Convert Markdown to HTML
    html = markdown.markdown(md_text)
    # Extract text from HTML
    text = BeautifulSoup(html, 'html.parser').get_text()
    # Count words
    return len(text.split())

In [72]:
tqdm.pandas()

rows = []
for group_name, group in tqdm(result_df.groupby(['intersection_id', 'state'])):
    subset = [c for c in group.columns if c not in ['lat', 'lng', 'group_id', 'zip']]
    ballot_text = format_and_display_ballot_info(group.drop_duplicates(['office.name', 'person.name']))
    
    races = [format_office(office, office_group)
             for office, office_group in group.groupby('office.name')
             if office_group.iloc[0].race_type != 'Ballot Measure']
    
    measures = [format_office(office, office_group)
                for office, office_group in group.groupby('office.name')
                if office_group.iloc[0].race_type == 'Ballot Measure']
    
    comp_races = [format_office(office, office_group)
                  for office, office_group in group.groupby('office.name')
                  if office_group.iloc[0].race_type != 'Ballot Measure' and len(office_group) > office_group.iloc[0].number_of_seats]
    
    non_partisan_races = [r.strip('# ').split('\n')[0] for r in races if 'nonpartisan' in r.lower()]
    ballot_length = len(ballot_text)
    word_count = count_markdown_words(ballot_text)
    
    row = {
        'intersection_id': group_name[0],
        'lat': group['lat'].iloc[0], 'lng': group['lng'].iloc[0],
        'zip': int(group['zip'].iloc[0]),
        'response': group['response'].iloc[0],
        'ballot_markdown': ballot_text, 
        'state_name': group_name[1],
        'county': group['county'].iloc[0],
        'district': group['district'].iloc[0],
        'unique_decisions':  group['office.name'].nunique(),
        'measures': measures,
        'races': races,
        'comp_races': comp_races,
        'non_partisan_races': non_partisan_races,
        "ballot_length": ballot_length,
        "word_count": word_count,
        'geometry': group['geometry'].iloc[0],
        'total_options': (group.race_type == 'Ballot Measure').sum() * 2 + (group.race_type != 'Ballot Measure').sum()
    }
    rows.append(row)

full_df = pd.DataFrame(rows).set_index(['intersection_id', 'state_name'])

100%|███████████████████████████████████████████████████████████████████| 50/50 [00:01<00:00, 38.58it/s]


In [74]:
full_df.tail(10)

,,lat,lng,zip,response,ballot_markdown,county,district,unique_decisions,measures,races,comp_races,non_partisan_races,ballot_length,word_count,geometry,total_options
intersection_id,state_name,,,,,,,,,,,,,,,,
187,Texas,33.881340,-101.598810,79250,"{""success"": true, ""data"": {""districts"": [{""id""...","# Ballot for Hale County, Texas\n\nElection Da...",Hale,TX-CD19,23,"[## Texas Proposition 1, Establish Special Fun...",[## Attorney General of Texas\n\n**Level:** St...,[## Attorney General of Texas\n\n**Level:** St...,[],6329,596,POLYGON ((-509937.0183781471 1229657.914430047...,67
188,Texas,33.983660,-101.993020,79021,"{""success"": true, ""data"": {""districts"": [{""id""...","# Ballot for Hale County, Texas\n\nElection Da...",Hale,TX-CD19,23,"[## Texas Proposition 1, Establish Special Fun...",[## Attorney General of Texas\n\n**Level:** St...,[## Attorney General of Texas\n\n**Level:** St...,[],6329,596,POLYGON ((-551068.5756636254 1231618.650399775...,67
189,Texas,34.287800,-101.894920,79032,"{""success"": true, ""data"": {""districts"": [{""id""...","# Ballot for Hale County, Texas\n\nElection Da...",Hale,TX-CD19,23,"[## Texas Proposition 1, Establish Special Fun...",[## Attorney General of Texas\n\n**Level:** St...,[## Attorney General of Texas\n\n**Level:** St...,[],6329,596,POLYGON ((-540454.0791680462 1266811.127153755...,67
190,Texas,33.857810,-101.882280,79311,"{""success"": true, ""data"": {""districts"": [{""id""...","# Ballot for Hale County, Texas\n\nElection Da...",Hale,TX-CD19,23,"[## Texas Proposition 1, Establish Special Fun...",[## Attorney General of Texas\n\n**Level:** St...,[## Attorney General of Texas\n\n**Level:** St...,[],6329,596,"POLYGON ((-525229.83892836 1211659.9064766727,...",67
191,Texas,33.956080,-101.949240,79073,"{""success"": true, ""data"": {""districts"": [{""id""...","# Ballot for Hale County, Texas\n\nElection Da...",Hale,TX-CD19,23,"[## Texas Proposition 1, Establish Special Fun...",[## Attorney General of Texas\n\n**Level:** St...,[## Attorney General of Texas\n\n**Level:** St...,[],6329,596,POLYGON ((-546156.7047345301 1228821.619732102...,67
192,Alabama,33.135830,-85.694478,36276,"{""success"": true, ""data"": {""districts"": [{""id""...","# Ballot for Clay County, Alabama\n\nElection ...",Clay,AL-CD03,2,[## Alabama Allow Judges to Deny Bail for Cert...,[],[],[],412,52,POLYGON ((956816.1426509152 1175354.1865688218...,4
193,Alabama,33.112564,-85.765995,36256,"{""success"": true, ""data"": {""districts"": [{""id""...","# Ballot for Clay County, Alabama\n\nElection ...",Clay,AL-CD03,2,[## Alabama Allow Judges to Deny Bail for Cert...,[],[],[],412,52,POLYGON ((949030.7837886392 1166290.6676511853...,4
194,Alabama,33.156300,-86.154560,35082,"{""success"": true, ""data"": {""districts"": [{""id""...","# Ballot for Clay County, Alabama\n\nElection ...",Clay,AL-CD03,2,[## Alabama Allow Judges to Deny Bail for Cert...,[],[],[],412,52,"POLYGON ((909418.2882089905 1164945.133038352,...",4
195,Alabama,33.118477,-86.163496,35150,"{""success"": true, ""data"": {""districts"": [{""id""...","# Ballot for Clay County, Alabama\n\nElection ...",Clay,AL-CD03,2,[## Alabama Allow Judges to Deny Bail for Cert...,[],[],[],412,52,POLYGON ((909475.3354828499 1164319.0129441076...,4


In [77]:
full_df

,,lat,lng,zip,response,ballot_markdown,county,district,unique_decisions,measures,races,comp_races,non_partisan_races,ballot_length,word_count,geometry,total_options
intersection_id,state_name,,,,,,,,,,,,,,,,
140,California,39.563360,-120.792790,95936,"{""success"": true, ""data"": {""districts"": [{""id""...","# Ballot for Sierra County, California\n\nElec...",Sierra,CA-CD03,1,"[## California Proposition 50, Use of Legislat...",[],[],[],214,24,POLYGON ((-2095264.5212082318 2111470.26092156...,2
141,California,39.433973,-120.972498,95960,"{""success"": true, ""data"": {""districts"": [{""id""...","# Ballot for Sierra County, California\n\nElec...",Sierra,CA-CD03,1,"[## California Proposition 50, Use of Legislat...",[],[],[],214,24,POLYGON ((-2103559.9321298925 2099602.53433268...,2
142,California,39.534820,-120.861590,95944,"{""success"": true, ""data"": {""districts"": [{""id""...","# Ballot for Sierra County, California\n\nElec...",Sierra,CA-CD03,1,"[## California Proposition 50, Use of Legislat...",[],[],[],214,24,POLYGON ((-2096173.572862002 2113442.188701270...,2
145,Tennessee,36.629042,-85.159074,38549,"{""success"": true, ""data"": {""districts"": [{""id""...","# Ballot for Clinton County, Tennessee\n\nElec...",Clinton,TN-CD01,3,[],[## Kentucky House of Representatives District...,[## U.S. House Kentucky District 1\n\n**Level:...,[],1730,134,MULTIPOLYGON (((957477.5037434566 1563339.0295...,14
146,Kentucky,36.670118,-85.264284,42717,"{""success"": true, ""data"": {""districts"": [{""id""...","# Ballot for Clinton County, Kentucky\n\nElect...",Clinton,KY-CD01,3,[],[## Kentucky House of Representatives District...,[## U.S. House Kentucky District 1\n\n**Level:...,[],1729,134,MULTIPOLYGON (((948198.6851217566 1562503.1105...,14
147,Kentucky,36.626036,-85.295355,42717,"{""success"": true, ""data"": {""districts"": [{""id""...","# Ballot for Clinton County, Kentucky\n\nElect...",Clinton,KY-CD01,3,[],[## Kentucky House of Representatives District...,[## U.S. House Kentucky District 1\n\n**Level:...,[],1729,134,MULTIPOLYGON (((946836.97126505 1562308.294266...,14
148,Kentucky,36.716344,-85.009947,42602,"{""success"": true, ""data"": {""districts"": [{""id""...","# Ballot for Clinton County, Kentucky\n\nElect...",Clinton,KY-CD05,2,[],[## U.S. House Kentucky District 5\n\n**Level:...,[## U.S. House Kentucky District 5\n\n**Level:...,[],1583,118,MULTIPOLYGON (((973730.4785105877 1564660.9135...,14
149,Kentucky,36.732010,-85.133240,42602,"{""success"": true, ""data"": {""districts"": [{""id""...","# Ballot for Clinton County, Kentucky\n\nElect...",Clinton,KY-CD01,3,[],[## Kentucky House of Representatives District...,[## U.S. House Kentucky District 1\n\n**Level:...,[],1729,134,POLYGON ((969991.0601767872 1577959.9489515154...,14
150,Kentucky,36.622390,-85.096303,42602,"{""success"": true, ""data"": {""districts"": [{""id""...","# Ballot for Clinton County, Kentucky\n\nElect...",Clinton,KY-CD01,3,[],[## Kentucky House of Representatives District...,[## U.S. House Kentucky District 1\n\n**Level:...,[],1729,134,POLYGON ((964046.5528060909 1563809.6571805691...,14


In [78]:
full_df['ballot_markdown']

intersection_id  state_name
140              California    # Ballot for Sierra County, California\n\nElec...
141              California    # Ballot for Sierra County, California\n\nElec...
142              California    # Ballot for Sierra County, California\n\nElec...
145              Tennessee     # Ballot for Clinton County, Tennessee\n\nElec...
146              Kentucky      # Ballot for Clinton County, Kentucky\n\nElect...
147              Kentucky      # Ballot for Clinton County, Kentucky\n\nElect...
148              Kentucky      # Ballot for Clinton County, Kentucky\n\nElect...
149              Kentucky      # Ballot for Clinton County, Kentucky\n\nElect...
150              Kentucky      # Ballot for Clinton County, Kentucky\n\nElect...
151              Kentucky      # Ballot for Clinton County, Kentucky\n\nElect...
152              Kentucky      # Ballot for Clinton County, Kentucky\n\nElect...
155              Ohio          # Ballot for Hancock County, Ohio\n\nElection ...


In [95]:
full_df.to_csv("data/2025/markdown_2025-22-25.csv")

In [80]:
search_string = '20251104'
filtered_df = full_df[full_df['ballot_markdown'].str.contains(search_string, case=False, na=False)]

print(filtered_df)

Empty DataFrame
Columns: [lat, lng, zip, response, ballot_markdown, county, district, unique_decisions, measures, races, comp_races, non_partisan_races, ballot_length, word_count, geometry, total_options]
Index: []


# Define readability calculator

Function to calculate Flesch-Kincaid Grade Level score for ballot text:
- Strips markdown formatting
- Returns reading grade level (higher score = more complex)

In [96]:
#thanks chagpt 

import re
from textstat import flesch_kincaid_grade
import ast  # to safely evaluate the stringified lists

def get_clean_prose(row):
    # Try to parse races and measures as Python lists
    try:
        races = ast.literal_eval(row['races']) if isinstance(row['races'], str) else []
        measures = ast.literal_eval(row['measures']) if isinstance(row['measures'], str) else []
    except Exception:
        races, measures = [], []
    
    # Combine and clean text
    combined = "\n".join(races + measures)
    # Remove markdown artifacts
    clean_text = re.sub(r'[#+*_`]', '', combined)
    return clean_text.strip()
    

def calculate_fk_from_cleaned(text):
    if not isinstance(text, str) or not re.search(r'[a-zA-Z]', text):
        return 0.0
    try:
        return round(flesch_kincaid_grade(text), 2)
    except Exception:
        return 0.0




In [87]:
#thasnks Obama! (chatgpt)
# def calculate_flesch_kincaid(text):
#     import re
#     from textstat import flesch_kincaid_grade

#     if not isinstance(text, str) or not re.search(r'[a-zA-Z]', text):
#         return 0.0  # or return 0.0 or np.nan
    
#     clean_text = re.sub(r'[#*_`]', '', text)
#     try:
#         return round(flesch_kincaid_grade(clean_text), 2)
#     except Exception:
#         return 0.0


In [81]:
# import re
# from textstat import flesch_kincaid_grade

# def calculate_flesch_kincaid(text):
#     if not isinstance(text, str) or not re.search(r'[a-zA-Z]', text):
#     return None  # or return 0.0 or np.nan
    
# # Remove any markdown formatting
#     clean_text = re.sub(r'[#*_`]', '', text)
    
#     # Calculate Flesch-Kincaid Grade Level
#     grade_level = flesch_kincaid_grade(clean_text)
    
#     return round(grade_level, 2)

In [104]:
#thanks Sam Altman
full_df["ballot_prose"] = full_df.apply(get_clean_prose, axis=1)
full_df["flesch_kincaid_grade"] = full_df["ballot_prose"].apply(calculate_fk_from_cleaned)
full_df["grade_level"] = full_df["flesch_kincaid_grade"].apply(lambda x: f"{x:.1f}")

In [88]:
# Apply the analysis to the DataFramerow['ballot_markdown'])
# full_df['flesch_kincaid_grade'] = full_df.ballot_markdown.progress_apply(calculate_flesch_kincaid)
# full_df['grade_level'] = full_df.flesch_kincaid_grade.apply(lambda x: f'{x:.1f}')

100%|█████████████████████████████████████████████████████████████████| 50/50 [00:00<00:00, 1910.08it/s]


In [105]:
full_df = full_df[full_df.ballot_markdown.notna()]

In [107]:
full_df['grade_level']

intersection_id  state_name
140              California    0.0
141              California    0.0
142              California    0.0
145              Tennessee     0.0
146              Kentucky      0.0
147              Kentucky      0.0
148              Kentucky      0.0
149              Kentucky      0.0
150              Kentucky      0.0
151              Kentucky      0.0
152              Kentucky      0.0
155              Ohio          0.0
157              Ohio          0.0
158              Ohio          0.0
159              Ohio          0.0
160              Ohio          0.0
162              Ohio          0.0
164              Ohio          0.0
165              Ohio          0.0
166              Ohio          0.0
167              Ohio          0.0
168              Ohio          0.0
169              Ohio          0.0
170              Ohio          0.0
171              Ohio          0.0
172              Ohio          0.0
173              Ohio          0.0
174              Ohio      

In [103]:
# pd.set_option('display.max_colwidth', None)
full_df['measures'].head(1)

intersection_id  state_name
140              California    [## California Proposition 50, Use of Legislative Congressional Redistricting Map Amendment (2025)\n\n**Type:** Ballot Measure\n**Level:** State\n\n---\n\n]
Name: measures, dtype: object

# Calculate complexity scores

Function to compute weighted ballot complexity scores based on multiple factors:
- Technical language and readability (Flesch-Kincaid)
- Ballot length and word count 
- Information density
- Number and complexity of decisions
- Non-partisan contest presence

Each factor is normalized against maximum values across all ballots.

In [28]:
full_df['avg_words_per_decision'] = full_df.word_count / full_df.unique_decisions
full_df['avg_options_per_decision'] = full_df.total_options / full_df.unique_decisions

In [29]:
def calculate_complexity_score(ballot_analysis, max_avg_options_per_decision, max_avg_words_per_decision, max_unique_decisions,
                               max_ballot_length, max_word_count, max_flesch_kincaid_grade):
    # Define weights for each indicator
    weights = {
        'ballot_length': 0.175,
        'word_count': 0.125,
        'avg_words_per_decision': 0.125,
        'unique_decisions': 0.15,
        'non_partisan_contests': 0.075,  # increased by 0.025
        'avg_options_per_decision': 0.125,
        'flesch_kincaid_grade': 0.125
    }    
    # Extract values with nested field access where necessary
    score = (
        (ballot_analysis['ballot_length'] * weights['ballot_length'] / max_ballot_length) +
        (ballot_analysis['word_count'] * weights['word_count'] / max_word_count) +
        (ballot_analysis['avg_words_per_decision'] * weights['avg_words_per_decision'] / max_avg_words_per_decision) +
        (ballot_analysis['avg_options_per_decision'] * weights['avg_options_per_decision'] / max_avg_options_per_decision) +
        (ballot_analysis['unique_decisions'] * weights['unique_decisions'] / max_unique_decisions) +
        (ballot_analysis['flesch_kincaid_grade'] * weights['flesch_kincaid_grade'] / max_flesch_kincaid_grade) +
        (int(not not ballot_analysis['non_partisan_races']) * weights['non_partisan_contests'])
    )
    
    return score

full_df['complexity_score'] = full_df.apply(
    calculate_complexity_score,
    max_ballot_length=full_df.ballot_length.max(),
    max_word_count=full_df.word_count.max(),
    max_avg_words_per_decision=full_df.avg_words_per_decision.max(),
    max_avg_options_per_decision=full_df.avg_options_per_decision.max(),
    max_unique_decisions=full_df.unique_decisions.max(),
    max_flesch_kincaid_grade=full_df.flesch_kincaid_grade.max(),
    axis=1
)

In [30]:
full_df['percentile'] = (full_df.complexity_score.rank() / len(full_df)) * 100

In [31]:
# # Sort and identify highest and lowest complexity ballots

full_df_sorted = full_df.reset_index().drop_duplicates('intersection_id').sort_values('complexity_score', ascending=False)
full_df_sorted['zip_count'] = full_df_sorted.zip.map(dict(full_df_sorted.zip.value_counts()))

highest_complexity = full_df_sorted.head(20000)
lowest_complexity = full_df_sorted.tail(50000)

highest_complexity[['zip', 'county', 'district', 'state_name', 'complexity_score', 'zip_count']].drop_duplicates(['state_name'], keep='first').head(10)
lowest_complexity[['zip', 'county', 'district', 'state_name', 'complexity_score', 'zip_count']].drop_duplicates(['state_name'], keep='last').tail(10)

,zip,county,district,state_name,complexity_score,zip_count
23439,77375,Harris,TX-CD02,Texas,0.684667,3
47032,48363,Oakland,MI-CD09,Michigan,0.580760,1
19974,92377,San Bernardino,CA-CD33,California,0.548130,1
5000,97227,Multnomah,OR-CD03,Oregon,0.519702,2
19745,59701,Silver Bow,MT-CD01,Montana,0.495234,6
31262,98283,Whatcom,WA-CD02,Washington,0.473855,2
36233,51544,Pottawattamie,IA-CD04,Iowa,0.462521,4
20105,68110,Douglas,NE-CD02,Nebraska,0.462221,1
15427,85035,Maricopa,AZ-CD03,Arizona,0.461927,2
9243,82431,Big Horn,WY-CD01,Wyoming,0.460147,1


,zip,county,district,state_name,complexity_score,zip_count
59198,36528,Mobile,AL-CD01,Alabama,0.189444,1
45347,40177,Hardin,KY-CD02,Kentucky,0.179346,5
58589,6706,New Haven,CT-CD03,Connecticut,0.178495,3
59319,4570,Lincoln,ME-CD01,Maine,0.177988,2
16718,3576,Coos,NH-CD02,New Hampshire,0.177982,2
57270,1944,Essex,MA-CD06,Massachusetts,0.165557,2
19714,29643,Anderson,SC-CD03,South Carolina,0.161379,2
59459,23440,Accomack,VA-CD02,Virginia,0.156886,1
1478,30250,Clayton,GA-CD13,Georgia,0.155076,1
37010,14468,Monroe,NY-CD25,New York,0.148877,1


# Define report generator

Function to create a markdown-formatted ballot complexity report with:
- Overall complexity score
- Decision complexity metrics (unique decisions, options, density)
- Language complexity (Flesch-Kincaid grade level)
- Ballot length statistics
- AI analysis disclaimer

In [32]:
def get_report_markdown(ballot_report, complexity_score):
    # Generate Markdown report with AI disclaimer
    report_md = f"""# Ballot Complexity Report
 
*This report provides an AI-assisted analysis of ballot complexity. Please note that this is a supplementary analysis and not a substitute for official election information.*

**This ballot is more complex than {str(round(ballot_report['percentile']) or 1).replace('100', '99')} percent of U.S. Ballots.**
|                         |                                 |                                          |
|-------------------------|---------------------------------|------------------------------------------|
| **Decision Complexity** | Number of Questions             | {ballot_report['unique_decisions']}      |
|                         | Average Words per Question      | {ballot_report['avg_words_per_decision']:.1f}|
|                         | Average Options per Question    | {ballot_report['avg_options_per_decision']:.1f}|
|                         | Number of Races                 | {len(ballot_report['races'])}|
|                         | Number of Competitive Races     | {len(ballot_report['comp_races'])}|
|                         | Number of Ballot Measures       | {len(ballot_report['measures'])}|
"""
    
    # Conditionally add Non-Partisan Contests
    if ballot_report['non_partisan_races']:
        report_md += "|                         | Non-Partisan Races  | " + ", ".join(ballot_report['non_partisan_races'][:3]) + " |\n"
    
    # Language Complexity Section
    report_md += f"""| **Language Complexity** | [Flesch-Kincaid Grade Level](https://ballotpedia.org/Ballot_measure_readability_scores,_2024#Flesch-Kincaid_Grade_Level)""" + \
    f"""| {ballot_report['flesch_kincaid_grade']}  years of education      |
"""
    # Length Section
    report_md += f"""| **Length**              | Ballot Length                 | {ballot_report['ballot_length']:,} characters       |
|                         | Word Count                     | {ballot_report['word_count']:,}                         |
"""
    return report_md.replace('.0', '')

In [33]:
pop_df = pd.read_csv('data/raw/zip_code_demographics.csv')
population_lookup = dict(zip(pop_df.zip.astype(str), pop_df.population))
state_id_lookup = dict(zip(pop_df.zip.astype(str), pop_df.state_id))

In [34]:
full_df['zip'] = full_df.zip.apply(lambda x: f'{int(x):05}').astype(str)
full_df['population'] = full_df.zip.map(population_lookup)
full_df = full_df.sort_values(by='population', ascending=False)

zip_lookup = full_df.reset_index().rename(columns={
    'county_name': 'county',
    'state_name': 'state',
})[['state', 'county', 'zip', 'population']]

zip_lookup = zip_lookup.sort_values(by='population', ascending=False).drop(columns=['population'])
zip_lookup = zip_lookup.drop_duplicates(['state', 'county', 'zip'])
zip_lookup.head()

zip_lookup.to_csv('data/processed/zip_lookup.csv', index=False)

,state,county,zip
0,Texas,Waller,77494
3,Texas,Fort Bend,77494
1,Texas,Harris,77494
6,Texas,Harris,77449
7,New York,Queens,11368


In [35]:
full_df['full_markdown'] = full_df.apply(
    lambda x: get_report_markdown(x, x.complexity_score), axis=1
) + '\n---\n' + full_df.ballot_markdown

In [36]:
COLUMNS = {
    'unique_decisions': 'unique_questions',
    'avg_words_per_decision': 'avg_words_per_question',
    'avg_options_per_decision': 'avg_options_per_question',
    'races': 'races',
    'measures': 'measures',
    'avg_words_per_decision': 'avg_words_per_question',
    'comp_races': 'competitive_races',
    'non_partisan_races': 'non_partisan_races',
    'flesch_kincaid_grade': 'flesch_kincaid_grade',
    'ballot_length': 'ballot_length',
    'word_count': 'word_count'
}

In [37]:
from collections import defaultdict

full_df['population'] = full_df.zip.map(population_lookup)
full_df['state_id'] = full_df.zip.map(state_id_lookup)
data_lookup = full_df.reset_index().sort_values(by='population', ascending=False).rename(columns={
    'county_name': 'county',
    'state_name': 'state',
    **COLUMNS
})[['state', 'district', 'county', 'zip', 'full_markdown', *COLUMNS.values()]].drop_duplicates(
    ['state', 'district', 'county', 'zip']
)

In [38]:
grouping = ['state', 'district', 'county', 'zip', 'full_markdown']
data_lookup = data_lookup.groupby(grouping).apply(
    lambda df: pd.Series({
        **df.drop(columns=grouping).to_dict(orient='records')[0],
        'measures': len(df.measures.iloc[0]),
        'races': len(df.races.iloc[0]),
        'competitive_races': len(df.competitive_races.iloc[0]),
        'non_partisan_races': df.non_partisan_races.iloc[0],
    })
).reset_index()
data_lookup['district'] = data_lookup.district.str.replace('00', '01')
data_lookup['district'] = data_lookup.district.str.replace('DC-CD98', '')

In [39]:
alert('Data Lookup Complete')

In [40]:
from IPython.display import display, Markdown
display(Markdown(data_lookup.full_markdown.sample(1).iloc[0]))

# Ballot Complexity Report
 
*This report provides an AI-assisted analysis of ballot complexity. Please note that this is a supplementary analysis and not a substitute for official election information.*

**This ballot is more complex than 75 percent of U.S. Ballots.**
|                         |                                 |                                          |
|-------------------------|---------------------------------|------------------------------------------|
| **Decision Complexity** | Number of Questions             | 20      |
|                         | Average Words per Question      | 26.4|
|                         | Average Options per Question    | 2.5|
|                         | Number of Races                 | 9|
|                         | Number of Competitive Races     | 9|
|                         | Number of Ballot Measures       | 11|
|                         | Non-Partisan Races  | Covelo Fire Protection District At-large, Mendocino Coast Health Care District Board At-large, Mendocino County Board of Education Trustee Area 3 |
| **Language Complexity** | [Flesch-Kincaid Grade Level](https://ballotpedia.org/Ballot_measure_readability_scores,_2024#Flesch-Kincaid_Grade_Level)| 23.5  years of education      |
| **Length**              | Ballot Length                 | 6,904 characters       |
|                         | Word Count                     | 527                         |

---
# Ballot for Mendocino County, California

Election Date: 2024-11-05

## President of the United States

**Level:** Federal
**Branch:** Executive
**Number of Seats:** 1

### Candidates:
- **Kamala D. Harris** (Democratic Party)
    - Running Mate: Tim Walz
    - [More Info](https://ballotpedia.org/Kamala_Harris)

- **Claudia De La Cruz** (Peace and Freedom Party)
    - Running Mate: Karina Garcia
    - [More Info](https://ballotpedia.org/Claudia_De_La_Cruz)

- **Chase Oliver** (Libertarian Party)
    - Running Mate: Mike ter Maat
    - [More Info](https://ballotpedia.org/Chase_Oliver)

- **Robert F. Kennedy Jr.** (American Independent Party)
    - Running Mate: Nicole Shanahan
    - [More Info](https://ballotpedia.org/Robert_F._Kennedy_Jr.)

- **Donald Trump** (Republican Party)
    - Running Mate: J.D. Vance
    - [More Info](https://ballotpedia.org/Donald_Trump)

- **Jill Stein** (Green Party)
    - Running Mate: Butch Ware
    - [More Info](https://ballotpedia.org/Jill_Stein)

---

## U.S. House California District 2

**Level:** Federal
**Branch:** Legislative
**Number of Seats:** 1

### Candidates:
- **Chris Coulombe** (Republican Party)
[More Info](https://ballotpedia.org/Chris_Coulombe)

- **Jared Huffman** (Democratic Party)
[More Info](https://ballotpedia.org/Jared_Huffman)

---

## U.S. Senate California

**Level:** Federal
**Branch:** Legislative
**Number of Seats:** 1

### Candidates:
- **Steve Garvey** (Republican Party)
[More Info](https://ballotpedia.org/Steve_Garvey_(California))

- **Adam Schiff** (Democratic Party)
[More Info](https://ballotpedia.org/Adam_Schiff)

---

## California State Assembly District 2

**Level:** State
**Branch:** Legislative
**Number of Seats:** 1

### Candidates:
- **Chris Rogers** (Democratic Party)
[More Info](https://ballotpedia.org/Chris_Rogers_(California))

- **Michael Greer** (Republican Party)
[More Info](https://ballotpedia.org/Michael_Greer_(California))

---

## Covelo Fire Protection District At-large

**Level:** Local
**Branch:** Legislative
**Number of Seats:** 3

### Candidates:
- **Cindy Nelson** (Nonpartisan)
[More Info](https://ballotpedia.org/Cindy_Nelson_(Covelo_Fire_Protection_District_At-large,_California,_candidate_2024))

- **Leanne G. Durham** (Nonpartisan)
[More Info](https://ballotpedia.org/Leanne_G._Durham_(Covelo_Fire_Protection_District_At-large,_California,_candidate_2024))

- **Edward Wilson** (Nonpartisan)
[More Info](https://ballotpedia.org/Edward_Wilson_(Covelo_Fire_Protection_District_At-large,_California,_candidate_2024))

- **Bryant Earl Hale** (Nonpartisan)
[More Info](https://ballotpedia.org/Bryant_Earl_Hale_(Covelo_Fire_Protection_District_At-large,_California,_candidate_2024))

- **Lindon A. Duke** (Nonpartisan)
[More Info](https://ballotpedia.org/Lindon_A._Duke_(Covelo_Fire_Protection_District_At-large,_California,_candidate_2024))

---

## Mendocino Coast Health Care District Board At-large

**Level:** Local
**Branch:** Legislative
**Number of Seats:** 2

### Candidates:
- **Lynn Finley** (Nonpartisan)
[More Info](https://ballotpedia.org/Lynn_Finley_(Mendocino_Coast_Health_Care_District_Board_At-large,_California,_candidate_2024))

- **Paul Katzeff** (Nonpartisan)
[More Info](https://ballotpedia.org/Paul_Katzeff_(Mendocino_Coast_Health_Care_District_Board_At-large,_California,_candidate_2024))

- **Mikael Blaisdell** (Nonpartisan)
[More Info](https://ballotpedia.org/Mikael_Blaisdell_(Mendocino_Coast_Health_Care_District_Board_At-large,_California,_candidate_2024))

- **Gabriel Quinn Maroney** (Nonpartisan)
[More Info](https://ballotpedia.org/Gabriel_Quinn_Maroney_(Mendocino_Coast_Health_Care_District_Board_At-large,_California,_candidate_2024))

---

## Mendocino County Board of Education Trustee Area 3

**Level:** Local
**Branch:** Legislative
**Number of Seats:** 1

### Candidates:
- **David R. Strock** (Nonpartisan)
[More Info](https://ballotpedia.org/David_R._Strock_(Mendocino_County_Board_of_Education_Trustee_Area_3,_California,_candidate_2024))

- **Michelle Hutchins** (Nonpartisan)
[More Info](https://ballotpedia.org/Michelle_Hutchins_(Mendocino_County_Board_of_Education_Trustee_Area_3,_California,_candidate_2024))

---

## Mendocino Unified School District school board Trustee Area 3

**Level:** Local
**Branch:** Legislative
**Number of Seats:** 1

### Candidates:
- **Jim Gagnon** (Nonpartisan)
[More Info](https://ballotpedia.org/Jim_Gagnon_(Mendocino_Unified_School_District_school_board_Trustee_Area_3,_California,_candidate_2024))

- **Michael Schaeffer** (Nonpartisan)
[More Info](https://ballotpedia.org/Michael_Schaeffer_(Mendocino_Unified_School_District_school_board_Trustee_Area_3,_California,_candidate_2024))

---

## Mendocino-Lake Community College District Governing Board Trustee Area 3

**Level:** Local
**Branch:** Legislative
**Number of Seats:** 1

### Candidates:
- **Gabriel Baca Meza** (Nonpartisan)
[More Info](https://ballotpedia.org/Gabriel_Baca_Meza_(Mendocino-Lake_Community_College_District_Governing_Board_Trustee_Area_3,_California,_candidate_2024))

- **Jay Epstein** (Nonpartisan)
[More Info](https://ballotpedia.org/Jay_Epstein_(Mendocino-Lake_Community_College_District_Governing_Board_Trustee_Area_3,_California,_candidate_2024))

---

## Albion-Little River Fire Protection District, California, Measure S, Parcel Unit Tax Measure (November 2024)

**Type:** Ballot Measure
**Level:** County

---

## California Proposition 2, Public Education Facilities Bond Measure (2024)

**Type:** Ballot Measure
**Level:** State

---

## California Proposition 3, Right to Marry and Repeal Proposition 8 Amendment (2024)

**Type:** Ballot Measure
**Level:** State

---

## California Proposition 32, $18 Minimum Wage Initiative (2024)

**Type:** Ballot Measure
**Level:** State

---

## California Proposition 33, Prohibit State Limitations on Local Rent Control Initiative (2024)

**Type:** Ballot Measure
**Level:** State

---

## California Proposition 34, Require Certain Participants in Medi-Cal Rx Program to Spend 98% of Revenues on Patient Care Initiative (2024)

**Type:** Ballot Measure
**Level:** State

---

## California Proposition 35, Managed Care Organization Tax Authorization Initiative (2024)

**Type:** Ballot Measure
**Level:** State

---

## California Proposition 36, Drug and Theft Crime Penalties and Treatment-Mandated Felonies Initiative (2024)

**Type:** Ballot Measure
**Level:** State

---

## California Proposition 4, Parks, Environment, Energy, and Water Bond Measure (2024)

**Type:** Ballot Measure
**Level:** State

---

## California Proposition 5, Lower Supermajority Requirement to 55% for Local Bond Measures to Fund Housing and Public Infrastructure Amendment (2024)

**Type:** Ballot Measure
**Level:** State

---

## California Proposition 6, Remove Involuntary Servitude as Punishment for Crime Amendment (2024)

**Type:** Ballot Measure
**Level:** State

---



In [41]:
from datetime import date

today = date.today().strftime("%Y%m%d")
data_lookup.to_csv(f'data/archive/data.csv')
data_lookup.to_csv(f'data/archive/data_{today}.csv')
for zip_code in tqdm(data_lookup.zip.unique()):
    data_lookup[data_lookup.zip == zip_code].to_csv(f'data/processed/zip_data_{zip_code}.csv'.lower().replace(' ', ''))

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 33698/33698 [00:56<00:00, 591.34it/s]


In [42]:
alert('Files Saved')

# Make maps

# TODO
[x] Percentile rank

[x] mf212mf@gmail.com

[x] bbg415bbg

[x] Percentile Rank for Complexity Score

[ ] Origin tweet -- edit and send out

[x] Fix the top ranking

# TODO

[ ] Surface outliers
[ ] Fix charts
[ ] Add literacy-readbility map
[ ] Add turnout map

In [49]:
alert("plots generated")

# Get NYTimes Data

True

# Perform repairs

Fix old reports without having to reprocess all the data.